# Advanced Problems with Solutions: Garbage Collection

This notebook contains advanced, solution-oriented exercises on Python garbage collection.

## Best practices used in this notebook
- Avoid relying on exact memory addresses or exact reference counts across Python implementations.
- Use `gc.collect()` before experiments to reduce noise from prior objects.
- Prefer observations such as "object is still tracked" or "object was collected" over brittle numeric assumptions.
- Use `weakref` where appropriate to observe object finalization without creating strong references.
- Keep experiments self-contained so each problem can be run independently.

> Note: This notebook is designed for CPython behavior and educational experimentation. Some low-level details may differ across Python versions and implementations.

In [1]:
import gc
import weakref
import ctypes

gc.collect()

def object_by_id(object_id):
    """Return whether an object with the specified id is currently tracked by the GC."""
    for obj in gc.get_objects():
        if id(obj) == object_id:
            return "Object exists"
    return "Not found"

def ref_count(address):
    """CPython-specific helper to inspect raw reference count by memory address."""
    return ctypes.c_long.from_address(address).value

print("GC enabled:", gc.isenabled())

GC enabled: True


## Problem 1 — Why reference counting alone is not enough

**Task:** Create two objects that reference each other, remove the external reference, and explain why the objects are still not immediately reclaimed when cyclic garbage collection is disabled.

### Solution outline
1. Disable the cyclic GC.
2. Create a cycle between two instances.
3. Remove the external name.
4. Show that the objects are still present.
5. Re-enable GC and collect.

**Key idea:** Pure reference counting cannot reclaim cycles, because each object in the cycle keeps the other's reference count above zero.

In [2]:
# Solution 1
class NodeA:
    def __init__(self):
        self.b = None

class NodeB:
    def __init__(self):
        self.a = None

gc.collect()
gc.disable()

a = NodeA()
b = NodeB()
a.b = b
b.a = a

a_id = id(a)
b_id = id(b)

print("Before deleting external refs:")
print("a:", object_by_id(a_id))
print("b:", object_by_id(b_id))

a = None
b = None

print("\nAfter deleting external refs, with gc disabled:")
print("a:", object_by_id(a_id))
print("b:", object_by_id(b_id))

gc.enable()
collected = gc.collect()

print("\nAfter re-enabling gc and collecting:")
print("Collected objects:", collected)
print("a:", object_by_id(a_id))
print("b:", object_by_id(b_id))

Before deleting external refs:
a: Object exists
b: Object exists

After deleting external refs, with gc disabled:
a: Object exists
b: Object exists

After re-enabling gc and collecting:
Collected objects: 2
a: Not found
b: Not found


### Discussion
- After `a = None` and `b = None`, there are no external names bound to the objects.
- However, the objects still reference each other.
- With cyclic GC disabled, they remain alive.
- Once cyclic GC is re-enabled and `gc.collect()` runs, the cycle becomes collectible.

---

## Problem 2 — Investigate tracking behavior

**Task:** Determine which objects the garbage collector tends to track and why that matters.

**Question:** Are all Python objects tracked by the cyclic GC?

**Expected idea:** No. Immutable atomic objects such as small integers and short strings often do not need cyclic GC tracking, while container objects usually do.

In [3]:
# Solution 2
gc.collect()

samples = {
    "small_int": 42,
    "string": "hello",
    "tuple_of_ints": (1, 2, 3),
    "list_of_ints": [1, 2, 3],
    "dict": {"x": 1},
    "set": {1, 2, 3}
}

for name, obj in samples.items():
    print(f"{name:15} tracked -> {gc.is_tracked(obj)}")

small_int       tracked -> False
string          tracked -> False
tuple_of_ints   tracked -> True
list_of_ints    tracked -> True
dict            tracked -> False
set             tracked -> True


### Discussion
- `gc.is_tracked(obj)` shows whether an object participates in cyclic garbage collection.
- Objects that cannot meaningfully participate in reference cycles do not always need tracking.
- This optimization reduces GC overhead.

---

## Problem 3 — Use `weakref` to observe collection safely

**Task:** Build a cycle, keep only weak references to the objects, and verify that the cycle disappears after collection.

**Why this is best practice:** A weak reference lets you observe an object without preventing its collection.

In [4]:
# Solution 3
class Person:
    def __init__(self, name):
        self.name = name
        self.partner = None

gc.collect()

p1 = Person("Alice")
p2 = Person("Bob")
p1.partner = p2
p2.partner = p1

w1 = weakref.ref(p1)
w2 = weakref.ref(p2)

print("Before deleting strong refs:")
print("w1() is None ->", w1() is None)
print("w2() is None ->", w2() is None)

p1 = None
p2 = None
gc.collect()

print("\nAfter deleting strong refs and collecting:")
print("w1() is None ->", w1() is None)
print("w2() is None ->", w2() is None)

Before deleting strong refs:
w1() is None -> False
w2() is None -> False

After deleting strong refs and collecting:
w1() is None -> True
w2() is None -> True


### Discussion
- The weak references do not keep the objects alive.
- Once all strong references are gone, the collector can reclaim the cycle.
- Weak references are especially useful in caches, graphs, observer patterns, and parent/child structures.

---

## Problem 4 — Compare a strong back-reference with a weak back-reference

**Task:** Model a parent-child relationship twice:
1. child stores a strong reference to parent
2. child stores a weak reference to parent

**Goal:** Explain why weak back-references are often preferable in object graphs.

In [5]:
# Solution 4
class StrongChild:
    def __init__(self, parent):
        self.parent = parent

class StrongParent:
    def __init__(self):
        self.child = StrongChild(self)

class WeakChild:
    def __init__(self, parent):
        self.parent = weakref.ref(parent)

class WeakParent:
    def __init__(self):
        self.child = WeakChild(self)

gc.collect()

sp = StrongParent()
sp_ref = weakref.ref(sp)
sp = None
gc.collect()

wp = WeakParent()
wp_ref = weakref.ref(wp)
wp = None
gc.collect()

print("Strong parent collected ->", sp_ref() is None)
print("Weak parent collected   ->", wp_ref() is None)

Strong parent collected -> True
Weak parent collected   -> True


### Discussion
- A strong back-reference can participate in cycles.
- A weak back-reference avoids turning a tree-like structure into a cycle.
- This design often reduces GC pressure and makes lifetimes easier to reason about.

---

## Problem 5 — Inspect collection statistics

**Task:** Use `gc.get_count()` and `gc.get_threshold()` to reason about Python's generational GC.

**Question:** What do these values represent, and why are there multiple generations?

**Answer idea:** Younger generations are collected more frequently because most objects die young.

In [6]:
# Solution 5
gc.collect()

print("GC thresholds:", gc.get_threshold())
print("GC counts before allocations:", gc.get_count())

temp_objects = []
for _ in range(10000):
    temp_objects.append([1, 2, 3])

print("GC counts after allocations:", gc.get_count())

temp_objects = None
collected = gc.collect()
print("Collected after cleanup:", collected)
print("GC counts after explicit collection:", gc.get_count())

GC thresholds: (2000, 10, 10)
GC counts before allocations: (45, 0, 0)
GC counts after allocations: (47, 5, 0)
Collected after cleanup: 0
GC counts after explicit collection: (22, 0, 0)


### Discussion
- `gc.get_threshold()` returns collection thresholds for generations.
- `gc.get_count()` returns allocation counters related to those generations.
- Python uses generational collection because short-lived objects are common.
- Tuning thresholds is rarely necessary in normal application code and should be driven by profiling.

---

## Problem 6 — Finalizers and cyclic garbage

**Task:** Explore how finalization interacts with collection.

**Important best-practice point:** Avoid putting critical resource management solely in `__del__`. Prefer context managers (`with`), explicit `close()`, or `weakref.finalize`.

Below is a safe modern pattern using `weakref.finalize`.

In [7]:
# Solution 6
messages = []

class ResourceHolder:
    def __init__(self, name):
        self.name = name
        self.other = None
        self._finalizer = weakref.finalize(self, messages.append, f"finalized:{name}")

gc.collect()

r1 = ResourceHolder("r1")
r2 = ResourceHolder("r2")
r1.other = r2
r2.other = r1

r1 = None
r2 = None
gc.collect()

print("Finalizer messages:", messages)

Finalizer messages: ['finalized:r1', 'finalized:r2']


### Discussion
- `weakref.finalize` is often safer than depending on `__del__`.
- It avoids several pitfalls of finalizers in complex object graphs.
- For files, sockets, locks, and DB connections, deterministic cleanup should be explicit.

---

## Problem 7 — Diagnose a realistic leak pattern

**Task:** A cache stores objects forever in a global list. Explain why this is not a GC bug and show a fix.

**Key concept:** If your program still holds strong references, the objects are not garbage. GC only reclaims unreachable objects.

In [8]:
# Solution 7
class Payload:
    def __init__(self, value):
        self.value = value

gc.collect()

cache = []
refs = []

for i in range(5):
    obj = Payload(i)
    refs.append(weakref.ref(obj))
    cache.append(obj)

print("Alive with cache retained:", [r() is not None for r in refs])

cache.clear()
obj = None
gc.collect()

print("Alive after cache cleared:", [r() is not None for r in refs])

Alive with cache retained: [True, True, True, True, True]
Alive after cache cleared: [False, False, False, False, False]


### Discussion
- This pattern is a logical memory leak, not a collector failure.
- The fix is to release the references, bound the cache, or use weak references where appropriate.
- Always distinguish between:
  1. unreachable garbage, and
  2. reachable-but-no-longer-useful objects.

---

## Problem 8 — Synthesize the rules

**Task:** Summarize when an object is collectible and when GC matters.

### Solution summary
An object can be reclaimed when it becomes unreachable from live program roots.

- If reference counting drops to zero immediately, CPython usually destroys it immediately.
- If objects form a cycle, reference counting alone may be insufficient.
- Cyclic GC detects and reclaims unreachable cycles.
- If live references still exist anywhere in the program, the object is not garbage.
- Weak references help observe or relate to objects without extending lifetime.
- Resource cleanup should usually be explicit, not left entirely to GC timing.

## Final takeaways
1. GC is about reachability, not whether you still care about an object.
2. Cycles are the main reason Python needs more than plain reference counting.
3. Weak references are an important design tool.
4. Deterministic cleanup should use context managers or explicit APIs.
5. Measure before tuning GC thresholds.